In [ ]:
!pip install datasets==3.6.0
!pip uninstall -y torchao
!pip install torchao>=0.16.0 --no-cache-dir

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 13.2 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

from datasets import Dataset
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from transformers import TrainingArguments, Trainer

from peft import get_peft_model, LoraConfig, TaskType

In [ ]:
reviews = pd.read_csv("imdb.csv").sample(500)
reviews.head()

,review,sentiment
9074,THE CAT O'NINE TAILS (Il Gatto a Nove Code) <b...,negative
4551,"In the ""goofs"" section for this film there's a...",positive
4512,"The plot starts out interesting, however, towa...",positive
5218,I like to think of myself as a bad movie conno...,negative
7043,Watching That Lady In Ermine I was wondering w...,negative


In [ ]:
reviews = reviews.rename(columns={"review": "text", "sentiment": "label"})
reviews['label'] = LabelEncoder().fit_transform(reviews['label'])
reviews.head()

,text,label
9074,THE CAT O'NINE TAILS (Il Gatto a Nove Code) <b...,0
4551,"In the ""goofs"" section for this film there's a...",1
4512,"The plot starts out interesting, however, towa...",1
5218,I like to think of myself as a bad movie conno...,0
7043,Watching That Lady In Ermine I was wondering w...,0


We see that 0 represents a negative review and 1 represents a positive review.

To help with this task, we're going to use the [Datasets library](https://huggingface.co/docs/datasets/en/index), which allows for working with various types of datasets, from text, to audio, to images.

First, convert the reviews DataFrame into a dataset object by using the [from_pandas function](https://huggingface.co/docs/datasets/v4.8.4/en/package_reference/main_classes#datasets.Dataset.from_pandas).

In [ ]:
ds = Dataset.from_pandas(reviews, preserve_index=False)

In [ ]:
ds

Dataset({
    features: ['text', 'label'],
    num_rows: 500
})

Once converted, we can perform a train/test split using a method of the Dataset object.

Peform an 80/20 train/test split using the [train_test_split method](https://huggingface.co/docs/datasets/v1.8.0/package_reference/main_classes.html#datasets.Dataset.train_test_split). Save the result back to the same Dataset object.

In [ ]:
ds = ds.train_test_split(test_size=0.20)

In [ ]:
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 400
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 100
    })
})

Now, extract the training and test portion into separate Datasets. Name these new datasets train_dataset and test_dataset, respectively.

In [ ]:
train_dataset = ds['train']
test_dataset = ds['test']

We're going to be working with a DistilBERT model, which means that we'll need to tokenize our input in the way that DistilBERT expects. For this, we can use the [DistilBertTokenizerFast](https://huggingface.co/docs/transformers/en/model_doc/distilbert?usage=AutoModel#transformers.DistilBertTokenizerFast) tokenizer.

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

Then we'll apply our tokenizer to the training and test datsets using the map method.

In [ ]:
train_dataset = train_dataset.map(lambda df: tokenizer(df['text'], padding="max_length", truncation=True), batched=True)
test_dataset = test_dataset.map(lambda df: tokenizer(df['text'], padding="max_length", truncation=True), batched=True)

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Finally, we'll set the format to torch so that we're working with [PyTorch](https://pytorch.org/) tensors and only extract the columns that we need.

In [ ]:
train_dataset

Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 400
})

In [ ]:
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

Now, let's load in the pretrained DistilBERT model.

### Part 1: Fine-tuning All Parameters

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Now, we need to set up a [Trainer](https://huggingface.co/docs/transformers/en/main_classes/trainer) for our model.

First, create a [TraininingArguments](https://huggingface.co/docs/transformers/v4.52.3/en/main_classes/trainer#transformers.TrainingArguments) object. Set num_train_epochs to 5, weight_decay to 0.01, and report_to = 'none'

In [ ]:
trainarg = TrainingArguments(
    num_train_epochs=5,
    weight_decay=0.01,
    report_to = 'none'
)

Finally, create a Trainer object using the model, the Training Arguments that you created, and with the train_dataset equal to train_dataset.

In [ ]:
trainer = Trainer(
    model=model,
    args=trainarg,
    train_dataset=train_dataset
)

Now, use the [train method](https://huggingface.co/docs/transformers/en/main_classes/trainer#transformers.Trainer.train) on your Trainer object.

In [ ]:
trainer.train()

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=250, training_loss=0.23625759887695313, metrics={'train_runtime': 104.8858, 'train_samples_per_second': 19.068, 'train_steps_per_second': 2.384, 'total_flos': 264934797312000.0, 'train_loss': 0.23625759887695313, 'epoch': 5.0})

Once the model has been fit, use the [predict method](https://huggingface.co/docs/transformers/en/main_classes/trainer#transformers.Trainer.predict) of the Trainer object to generate a set of predictions on the test_dataset.

In [ ]:
predictions = trainer.predict(test_dataset)

Extract the actual test labels and the predicted labels from the predictions. Note that both the true labels and predicted probabilites are contained as an attribute of the predictions.

In [ ]:
pred_labels = np.argmax(predictions.predictions, axis=-1)

In [ ]:
actual_labels = predictions.label_ids

Finally, look the confusion matrix and classification report for these predictions.

In [ ]:
print(confusion_matrix(actual_labels, pred_labels))

[[42  9]
 [ 7 42]]


In [ ]:
print(classification_report(actual_labels, pred_labels))

              precision    recall  f1-score   support

           0       0.86      0.82      0.84        51
           1       0.82      0.86      0.84        49

    accuracy                           0.84       100
   macro avg       0.84      0.84      0.84       100
weighted avg       0.84      0.84      0.84       100



### Part 2: Training Only a Subset of the Parameters

Let's first reload the pretrained distilbert model.

Then, we'll make none of the parameters trainable by setting the `requires_grad` attribute to False.

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased")

for param in model.distilbert.parameters():
    param.requires_grad = False

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Then, we'll go back and make the last 2 layers trainable.

In [ ]:
for i in [4, 5]:
    for param in model.distilbert.transformer.layer[i].parameters():
        param.requires_grad = True

We'll now set up TrainingArguments and a Trainer as before to train the model.

In [ ]:
training_args = TrainingArguments(
    num_train_epochs=5,
    weight_decay=0.01,
    report_to = 'none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

trainer.train()

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=250, training_loss=0.27714776611328124, metrics={'train_runtime': 55.85, 'train_samples_per_second': 35.81, 'train_steps_per_second': 4.476, 'total_flos': 264934797312000.0, 'train_loss': 0.27714776611328124, 'epoch': 5.0})

How do the results of fine-tuning only the last two layers compare to fine-tuning all layers? How does the training time compare?

In [ ]:
predictions = trainer.predict(test_dataset)
pred_labels = np.argmax(predictions.predictions, axis=-1)
actual_labels = predictions.label_ids
print(confusion_matrix(actual_labels, pred_labels))
print(classification_report(actual_labels, pred_labels))

[[43  8]
 [ 7 42]]
              precision    recall  f1-score   support

           0       0.86      0.84      0.85        51
           1       0.84      0.86      0.85        49

    accuracy                           0.85       100
   macro avg       0.85      0.85      0.85       100
weighted avg       0.85      0.85      0.85       100



Fine tuning just the last 2 layers reduced training time by at least half. It also improved the recall and f1-score for negative reviews, and improved the precision and f1-score for positive reviews. The model picked up 1 additional correct negative review.

### Part 3: Parameter-Efficient Fine-Tuning

Now, let's see how we can use the [peft library](https://huggingface.co/docs/peft/en/index) to more efficiently fine-tune our model.

First, we'll re-initalize the model.

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Create a [LoraConfig object](https://huggingface.co/docs/peft/package_reference/lora#peft.LoraConfig) where you set the Lora attention dimension to 8, the lora_alpha to 16, the lora_dropout to 0.1, the target_modules to "q_lin" and "v_lin" (these are the "query" and "value" projections), and set the task_type to TaskType.SEQ_CLS.

In [ ]:
loraconf = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],
    task_type=TaskType.SEQ_CLS
)

Now, use the [get_peft_model function](https://huggingface.co/docs/peft/v0.15.0/en/package_reference/peft_model#peft.get_peft_model) to create the Lora model pass in the distilbert model and the LoraConfig object that you created.

In [ ]:
loramodel = get_peft_model(model, loraconf)

How many trainable parameters does the resulting model have? Hint: you can use the [print_trainable_parameters function](https://huggingface.co/docs/peft/v0.15.0/en/package_reference/peft_model#peft.PeftModel.print_trainable_parameters).

In [ ]:
loramodel.print_trainable_parameters()

trainable params: 739,586 || all params: 67,694,596 || trainable%: 1.0925


We'll again set up the same TrainingArguments and create our Trainer object to train the model.

In [ ]:
training_args = TrainingArguments(
    num_train_epochs=5,
    weight_decay=0.01,
    report_to = 'none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

trainer.train()

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=250, training_loss=0.5772886352539063, metrics={'train_runtime': 79.2521, 'train_samples_per_second': 25.236, 'train_steps_per_second': 3.154, 'total_flos': 269478813696000.0, 'train_loss': 0.5772886352539063, 'epoch': 5.0})

How does the performance of the lora model compare to the previous two? How does the training time compare?

In [ ]:
predictions = trainer.predict(test_dataset)
pred_labels = np.argmax(predictions.predictions, axis=-1)
actual_labels = predictions.label_ids
print(confusion_matrix(actual_labels, pred_labels))
print(classification_report(actual_labels, pred_labels))

[[41 10]
 [10 39]]
              precision    recall  f1-score   support

           0       0.80      0.80      0.80        51
           1       0.80      0.80      0.80        49

    accuracy                           0.80       100
   macro avg       0.80      0.80      0.80       100
weighted avg       0.80      0.80      0.80       100



Training time was better than when fine tuning all parameters but worse than just fine tuning the last 2 layers. Compared to fine tuning just the last 2 layers, this model incorrectly classified 2 more negative reviews and incorrectly classified 3 more positive reviews, leading to worse precision, recall, and f1-scores across the 2 classes.